# Data Classification - Structured Data

This notebook represents a simple example of the utilization of the READI framework. In this series of example we will focus on data classification.
We refer to data classification as the ability to assign semantic to values represented in an input dataset at various granularity, would it be a column in a tabular dataset or spans within an unstructured document.
In particular, we will here focus on structured data classification, thus the classification of data with predefined and explicit structure. This include CSV, JSON, XML, database tables and similar formats.


## Installation

The installation of the framework can be performed simply via pip install, as here demonstrated:

## Example dataset

Structured datasets can easily be processed as pandas DataFrame. For example, let us consider a fictitious medical datasets as follows:

In [ ]:
import pandas as pd

example_dataset = pd.read_csv("./healthcare-dataset.csv", header=None)

example_dataset

This dataset represents a typical demographic table for a medical dataset. Note that this specific example is a synthetic dataset, but it reflects values and distributions observed in various client engagement.
One might notice that the dataset does not have a header, or metadata. This is a common scenario when dealing with dataset from dubious sources, or even when the dataset is extracted from enterprise knowledge bases that evolved through time. In particular, in the latter case it is not uncommon that specific knowledge about the content of various database columns is uncertain to current employees.

In [ ]:
from risk_assessment.classification import DatasetClassification, DatasetClassificationConfiguration
from risk_assessment.classification.identifiers import (
    SSN,
    URI,
    DateTime,
    Email,
    Etnicity,
    Gender,
    ICDv9,
    MaritalStatus,
    Name,
    Person,
    Religion,
    Surname,
    YearOfBirth,
)

configuration = DatasetClassificationConfiguration(
    identifiers=[
        DateTime(),
        Email(),
        Etnicity(),
        Email(),
        Gender(),
        ICDv9(),
        MaritalStatus(),
        Name(),
        Person(),
        Religion(),
        SSN(),
        Surname(),
        URI(),
        YearOfBirth(),
    ]
)  # this configuration specifies only a list of identifiers we are interested to detect

table_detector = DatasetClassification(configuration)

In [ ]:
report = table_detector.classify(example_dataset)

The report variable will contain the guessed types according to the provided configuration.
Note that the report contains two main components: `best_type`, which contains the best guess of the types associated to each column based on the information collected.
This is based on relative priority, reliability, and accuracy of the identifiers used.
Moreover, we remark that if the `DatasetClassifier` was not able to make an informed decision, it will select `UNKNOWN` as default type instead of randomly selecting one of the identified types.
Unless otherwise instructed from the configuration, which allows a user to specify various changes to the behavior, including minimum thresholds, relative priority and more. See [DatasetClassifierConfiguration](../risk_assessment/classification/__init__.py#L41)

In [ ]:
report.best_types

And `reports`, which contains the raw values from the detection process. This field is mostly meant as a way to debug, or further disambiguate unknown fields.
Note that for each column on `report.reports` the numerical values per time represent the ration of hits over the entire dataset, not the relative ratio.

In [ ]:
report.reports

### Extension with custom identifiers

The framework is fully customizable. For example, one might want to define a custom identifier to detect domain specific information. More concretely, let us assume that the first column of the previous dataset contains a Patient ID, that in the local hospital has the format of `P` followed by 9 digits. We can create an identifier based on such regular expression.

In [ ]:
import re

from risk_assessment.classification.identifiers import RegexIdentifier

patient_id = RegexIdentifier("PatientID", [re.compile(r"^P0000[263741895][0-9][0-9][0-9][0-9]$")])

This identiifer can now be used in our pipeline:

In [ ]:
configuration = DatasetClassificationConfiguration(
    identifiers=[
        DateTime(),
        Email(),
        Etnicity(),
        Email(),
        Gender(),
        ICDv9(),
        MaritalStatus(),
        Name(),
        Person(),
        Religion(),
        SSN(),
        Surname(),
        URI(),
        YearOfBirth(),
        patient_id,  # note the custom identifier
    ]
)

DatasetClassification(configuration).classify(example_dataset).best_types